# Notebook 07 — Model Validation and Comparison

Notebook 06 identified two remaining log-response candidate specifications:

- the full eight-block model including `Coupons`;
- the reduced seven-block model excluding `Coupons`.

The eight-block specification is provisionally preferred, while refined weighted least squares (WLS) is retained as the primary estimation approach.

This notebook evaluates both candidate specifications using the held-out validation sample. Validation focuses on out-of-sample prediction performance, coefficient stability, and model parsimony.

No new candidate specifications, transformations, or variance-function refinements are introduced using the validation sample. The validation sample is used only to assess the two pre-specified candidates and support the final model choice.

## 1. Reconstruct the Model-Building and Validation Samples

The original 50/50 split and categorical grouping rules are reproduced exactly.

All category definitions continue to be determined exclusively from the model-building sample.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import statsmodels.formula.api as smf

In [2]:
PROJECT_DIR = Path.cwd().resolve()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = (
    PROCESSED_DATA_DIR
    / "db1c_ticket_prepared_202606.parquet"
)

df = pd.read_parquet(DATA_FILE)

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows:    90,086
Columns: 16


### 1.1 Reproduce the 50/50 Split

In [3]:
SPLIT_SEED = 42
TRAIN_SHARE = 0.50

rng = np.random.default_rng(SPLIT_SEED)

random_order = rng.permutation(len(df))
split_point = int(TRAIN_SHARE * len(df))

train_idx = random_order[:split_point]
validation_idx = random_order[split_point:]

model_df = (
    df.iloc[train_idx]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    df.iloc[validation_idx]
    .copy()
    .reset_index(drop=True)
)

print(f"Model-building sample: {len(model_df):,}")
print(f"Validation sample:     {len(validation_df):,}")

Model-building sample: 45,043
Validation sample:     45,043


### 1.2 Reproduce Categorical Grouping

In [4]:
MIN_CATEGORY_COUNT = 100

carrier_counts = model_df["RpCarrier"].value_counts()

major_carriers = carrier_counts[
    carrier_counts >= MIN_CATEGORY_COUNT
].index

model_df["RpCarrier_Model"] = model_df["RpCarrier"].where(
    model_df["RpCarrier"].isin(major_carriers),
    "Other"
)

validation_df["RpCarrier_Model"] = validation_df["RpCarrier"].where(
    validation_df["RpCarrier"].isin(major_carriers),
    "Other"
)

state_counts = model_df["OriginState"].value_counts()

major_states = state_counts[
    state_counts >= MIN_CATEGORY_COUNT
].index

model_df["OriginState_Model"] = model_df["OriginState"].where(
    model_df["OriginState"].isin(major_states),
    "Other"
)

validation_df["OriginState_Model"] = validation_df["OriginState"].where(
    validation_df["OriginState"].isin(major_states),
    "Other"
)

print(
    f"Carrier model categories: "
    f"{model_df['RpCarrier_Model'].nunique()}"
)

print(
    f"Origin-state model categories: "
    f"{model_df['OriginState_Model'].nunique()}"
)

Carrier model categories: 12
Origin-state model categories: 43


#### Interpretation

The original model-building and validation samples are reproduced successfully.

Category definitions are determined exclusively from the model-building sample, preserving the validation sample for independent model assessment.

## 2. Reconstruct the Candidate Training Models

The two candidate log-response specifications are reconstructed using the model-building sample.

The candidates differ only in the inclusion of `Coupons`. Both are estimated using the same refined WLS procedure so that subsequent validation isolates the contribution of the additional predictor rather than differences in estimation method.

### 2.1 Prepare the Log Response

In [5]:
model_df["LogTotalAmount"] = np.log(
    model_df["TotalAmount"]
)

validation_df["LogTotalAmount"] = np.log(
    validation_df["TotalAmount"]
)

### 2.2 Define the Candidate Specifications

In [6]:
full_log_formula = """
LogTotalAmount ~
    TotalDistance
    + Coupons
    + C(RoundTrip, Treatment(reference=0))
    + C(OnLine, Treatment(reference=1))
    + C(PurchaseWindowGroup, Treatment(reference='21AP'))
    + C(RpCarrier_Model, Treatment(reference='AA'))
    + C(ItinGeoType, Treatment(reference=2))
    + C(OriginState_Model, Treatment(reference='CA'))
"""

reduced_log_formula = """
LogTotalAmount ~
    TotalDistance
    + C(RoundTrip, Treatment(reference=0))
    + C(OnLine, Treatment(reference=1))
    + C(PurchaseWindowGroup, Treatment(reference='21AP'))
    + C(RpCarrier_Model, Treatment(reference='AA'))
    + C(ItinGeoType, Treatment(reference=2))
    + C(OriginState_Model, Treatment(reference='CA'))
"""

In [7]:
full_log_model = smf.ols(
    formula=full_log_formula,
    data=model_df
).fit()

reduced_log_model = smf.ols(
    formula=reduced_log_formula,
    data=model_df
).fit()

print(
    f"Eight-block parameters: "
    f"{int(full_log_model.df_model + 1)}"
)

print(
    f"Seven-block parameters: "
    f"{int(reduced_log_model.df_model + 1)}"
)

Eight-block parameters: 61
Seven-block parameters: 60


### 2.3 Construct Refined WLS Weights

The refined variance-function form identified in Notebook 06 is applied consistently to both candidate specifications.

For each candidate, absolute log-OLS residuals are modelled using fitted log fare, its squared term, `TotalDistance`, `RoundTrip`, and `OnLine`. The functional form is fixed in advance; only its coefficients are estimated from the model-building sample.

In [8]:
full_wls_data = model_df.copy()

full_wls_data["Fitted_LogFare"] = (
    full_log_model.fittedvalues.to_numpy()
)

full_wls_data["Abs_Residual"] = np.abs(
    full_log_model.resid.to_numpy()
)

full_sd_model = smf.ols(
    """
    Abs_Residual ~
        Fitted_LogFare
        + I(Fitted_LogFare**2)
        + TotalDistance
        + C(RoundTrip, Treatment(reference=0))
        + C(OnLine, Treatment(reference=1))
    """,
    data=full_wls_data
).fit()

full_sd = full_sd_model.predict(
    full_wls_data
)

full_wls_data["WLS_Weight"] = (
    1 / full_sd**2
)

full_wls_model = smf.wls(
    formula=full_log_formula,
    data=full_wls_data,
    weights=full_wls_data["WLS_Weight"]
).fit()

In [9]:
reduced_wls_data = model_df.copy()

reduced_wls_data["Fitted_LogFare"] = (
    reduced_log_model.fittedvalues.to_numpy()
)

reduced_wls_data["Abs_Residual"] = np.abs(
    reduced_log_model.resid.to_numpy()
)

reduced_sd_model = smf.ols(
    """
    Abs_Residual ~
        Fitted_LogFare
        + I(Fitted_LogFare**2)
        + TotalDistance
        + C(RoundTrip, Treatment(reference=0))
        + C(OnLine, Treatment(reference=1))
    """,
    data=reduced_wls_data
).fit()

reduced_sd = reduced_sd_model.predict(
    reduced_wls_data
)

reduced_wls_data["WLS_Weight"] = (
    1 / reduced_sd**2
)

reduced_wls_model = smf.wls(
    formula=reduced_log_formula,
    data=reduced_wls_data,
    weights=reduced_wls_data["WLS_Weight"]
).fit()

In [10]:
candidate_check = pd.DataFrame({
    "Candidate": [
        "Eight-block",
        "Seven-block"
    ],
    "Parameters": [
        int(full_wls_model.df_model + 1),
        int(reduced_wls_model.df_model + 1)
    ],
    "Observations": [
        int(full_wls_model.nobs),
        int(reduced_wls_model.nobs)
    ]
})

display(candidate_check)

,Candidate,Parameters,Observations
0,Eight-block,61,45043
1,Seven-block,60,45043


## 3. Apply the Candidate Models to the Validation Sample

Each candidate model is now applied to the held-out validation sample.

Out-of-sample predictions are generated using coefficients estimated from the model-building sample. For WLS, the corresponding training-estimated variance function is transferred to the validation observations without re-estimation.

### 3.1 Generate Validation Weights

In [11]:
full_validation_data = validation_df.copy()

full_validation_data["Fitted_LogFare"] = (
    full_log_model.predict(full_validation_data)
)

full_validation_sd = full_sd_model.predict(
    full_validation_data
)

full_validation_data["WLS_Weight"] = (
    1 / full_validation_sd**2
)

print(
    f"Eight-block validation SD range: "
    f"{full_validation_sd.min():.6f} to "
    f"{full_validation_sd.max():.6f}"
)

print(
    f"Eight-block max / min weight ratio: "
    f"{full_validation_data['WLS_Weight'].max() / full_validation_data['WLS_Weight'].min():.2f}"
)

Eight-block validation SD range: 0.283417 to 1.114389
Eight-block max / min weight ratio: 15.46


In [12]:
reduced_validation_data = validation_df.copy()

reduced_validation_data["Fitted_LogFare"] = (
    reduced_log_model.predict(reduced_validation_data)
)

reduced_validation_sd = reduced_sd_model.predict(
    reduced_validation_data
)

reduced_validation_data["WLS_Weight"] = (
    1 / reduced_validation_sd**2
)

print(
    f"Seven-block validation SD range: "
    f"{reduced_validation_sd.min():.6f} to "
    f"{reduced_validation_sd.max():.6f}"
)

print(
    f"Seven-block max / min weight ratio: "
    f"{reduced_validation_data['WLS_Weight'].max() / reduced_validation_data['WLS_Weight'].min():.2f}"
)

Seven-block validation SD range: 0.283540 to 1.106107
Seven-block max / min weight ratio: 15.22


#### Interpretation

Validation weights are generated using variance functions estimated exclusively from the model-building sample.

No validation residuals are used to modify the weighting schemes, preserving the independence of the holdout assessment.

### 3.2 Generate Out-of-Sample Predictions

In [13]:
full_validation_pred = (
    full_wls_model.predict(
        full_validation_data
    )
)

reduced_validation_pred = (
    reduced_wls_model.predict(
        reduced_validation_data
    )
)

validation_predictions = pd.DataFrame({
    "LogTotalAmount": validation_df[
        "LogTotalAmount"
    ],
    "Pred_Eight_Block": full_validation_pred,
    "Pred_Seven_Block": reduced_validation_pred
})

display(
    validation_predictions.head()
)

,LogTotalAmount,Pred_Eight_Block,Pred_Seven_Block
0,6.637743,6.292711,6.296388
1,5.739632,5.660987,5.662725
2,5.664348,5.637964,5.643472
3,5.812786,6.287256,6.285487
4,5.957934,6.320375,6.319191


## 4. Out-of-Sample Prediction Performance

Prediction performance is evaluated using mean squared prediction error:

$$
MSPR
=
\frac{
\sum_{i=1}^{n^*}
(Y_i-\hat Y_i)^2
}{
n^*
},
$$

where $\hat Y_i$ is obtained from the model fitted using the model-building sample.

Lower MSPR indicates better out-of-sample predictive performance.

### 4.1 Calculate Validation MSPR

In [14]:
validation_actual = validation_df[
    "LogTotalAmount"
].to_numpy()

full_mspr = np.mean(
    (
        validation_actual
        - full_validation_pred.to_numpy()
    ) ** 2
)

reduced_mspr = np.mean(
    (
        validation_actual
        - reduced_validation_pred.to_numpy()
    ) ** 2
)

print(
    f"Eight-block MSPR: {full_mspr:.6f}"
)

print(
    f"Seven-block MSPR: {reduced_mspr:.6f}"
)

Eight-block MSPR: 0.206064
Seven-block MSPR: 0.206112


In [15]:
mspr_comparison = pd.DataFrame({
    "Candidate": [
        "Eight-block",
        "Seven-block"
    ],
    "Parameters": [
        int(full_wls_model.df_model + 1),
        int(reduced_wls_model.df_model + 1)
    ],
    "Validation_MSPR": [
        full_mspr,
        reduced_mspr
    ]
})

display(
    mspr_comparison.round(6)
)

,Candidate,Parameters,Validation_MSPR
0,Eight-block,61,0.206064
1,Seven-block,60,0.206112


#### Interpretation

Both candidate specifications produce nearly identical out-of-sample prediction performance.

The eight-block model has a slightly lower MSPR of 0.206064 compared with 0.206112 for the seven-block model. The difference is very small, indicating that including `Coupons` provides only a marginal improvement in predictive accuracy.

Prediction performance alone therefore does not clearly distinguish the two candidates. The final choice should also consider the relationship between training and validation error, coefficient stability, and parsimony.

### 4.2 Compare Training MSE with Validation MSPR

Training prediction error is compared with validation MSPR to assess whether each candidate maintains similar predictive performance when applied to unseen observations.

For consistency across the two WLS specifications, training MSE is calculated using ordinary squared prediction errors on the log-fare scale:

$$
MSE_{\text{train}}
=
\frac{
\sum_{i=1}^{n}
(Y_i-\hat Y_i)^2
}{
n-p
}.
$$

In [16]:
train_actual = model_df[
    "LogTotalAmount"
].to_numpy()

full_train_pred = (
    full_wls_model.predict(
        full_wls_data
    )
    .to_numpy()
)

reduced_train_pred = (
    reduced_wls_model.predict(
        reduced_wls_data
    )
    .to_numpy()
)

p_full = int(
    full_wls_model.df_model + 1
)

p_reduced = int(
    reduced_wls_model.df_model + 1
)

n_train = len(model_df)

full_train_sse = np.sum(
    (train_actual - full_train_pred) ** 2
)

reduced_train_sse = np.sum(
    (train_actual - reduced_train_pred) ** 2
)

full_train_mse = (
    full_train_sse
    / (n_train - p_full)
)

reduced_train_mse = (
    reduced_train_sse
    / (n_train - p_reduced)
)

print(
    f"Eight-block training MSE: "
    f"{full_train_mse:.6f}"
)

print(
    f"Seven-block training MSE: "
    f"{reduced_train_mse:.6f}"
)

Eight-block training MSE: 0.203005
Seven-block training MSE: 0.203042


In [17]:
prediction_validation = pd.DataFrame({
    "Candidate": [
        "Eight-block",
        "Seven-block"
    ],
    "Training_MSE": [
        full_train_mse,
        reduced_train_mse
    ],
    "Validation_MSPR": [
        full_mspr,
        reduced_mspr
    ]
})

prediction_validation["MSPR_MSE_Ratio"] = (
    prediction_validation["Validation_MSPR"]
    / prediction_validation["Training_MSE"]
)

prediction_validation[
    "MSPR_MSE_Gap_Percent"
] = (
    (
        prediction_validation[
            "MSPR_MSE_Ratio"
        ] - 1
    ) * 100
)

display(
    prediction_validation.round(6)
)

,Candidate,Training_MSE,Validation_MSPR,MSPR_MSE_Ratio,MSPR_MSE_Gap_Percent
0,Eight-block,0.203005,0.206064,1.015071,1.507110
1,Seven-block,0.203042,0.206112,1.015122,1.512249


#### Interpretation

Both candidate specifications show stable out-of-sample performance.

For the eight-block model, validation MSPR is approximately 1.51% higher than the training MSE. The seven-block model shows a nearly identical increase of approximately 1.51%.

The small training–validation gaps indicate little deterioration in predictive performance on unseen observations and provide no evidence of material overfitting.

The eight-block model performs marginally better on both training MSE and validation MSPR, but the differences between the two candidates are very small. Prediction error alone therefore does not provide a strong basis for choosing between them.

## 5. Coefficient Stability Across Samples

Each candidate specification is re-estimated on the validation sample using the corresponding training-derived WLS weights.

The validation estimates are compared with the model-building estimates to assess whether the estimated relationships remain stable in sign and magnitude across independent samples.

### 5.1 Fit the Candidate Models on the Validation Sample

In [18]:
full_validation_model = smf.wls(
    formula=full_log_formula,
    data=full_validation_data,
    weights=full_validation_data["WLS_Weight"]
).fit()

reduced_validation_model = smf.wls(
    formula=reduced_log_formula,
    data=reduced_validation_data,
    weights=reduced_validation_data["WLS_Weight"]
).fit()

print(
    f"Eight-block validation observations: "
    f"{int(full_validation_model.nobs):,}"
)

print(
    f"Seven-block validation observations: "
    f"{int(reduced_validation_model.nobs):,}"
)

Eight-block validation observations: 45,043
Seven-block validation observations: 45,043


### 5.2 Eight-Block Coefficient Stability

In [19]:
full_coefficient_stability = pd.DataFrame({
    "Coefficient": full_wls_model.params.index,
    "Training_Estimate": full_wls_model.params.values,
    "Validation_Estimate": (
        full_validation_model.params
        .reindex(full_wls_model.params.index)
        .values
    ),
    "Training_SE": full_wls_model.bse.values,
    "Validation_SE": (
        full_validation_model.bse
        .reindex(full_wls_model.params.index)
        .values
    )
})

full_coefficient_stability[
    "Estimate_Difference"
] = (
    full_coefficient_stability["Validation_Estimate"]
    - full_coefficient_stability["Training_Estimate"]
)

full_coefficient_stability[
    "Abs_Estimate_Difference"
] = (
    full_coefficient_stability[
        "Estimate_Difference"
    ].abs()
)

full_coefficient_stability["Sign_Change"] = (
    np.sign(
        full_coefficient_stability["Training_Estimate"]
    )
    !=
    np.sign(
        full_coefficient_stability["Validation_Estimate"]
    )
)

display(
    full_coefficient_stability
    .sort_values(
        "Abs_Estimate_Difference",
        ascending=False
    )
    .head(10)
)

,Coefficient,Training_Estimate,Validation_Estimate,Training_SE,Validation_SE,Estimate_Difference,Abs_Estimate_Difference,Sign_Change
15,"C(RpCarrier_Model, Treatment(reference='AA'))[...",-1.018505,-0.951932,0.040426,0.043192,0.066573,0.066573,False
11,"C(RpCarrier_Model, Treatment(reference='AA'))[...",-0.436701,-0.371955,0.035177,0.037207,0.064746,0.064746,False
17,"C(OriginState_Model, Treatment(reference='CA')...",0.174534,0.112730,0.027430,0.027470,-0.061804,0.061804,False
58,"C(OriginState_Model, Treatment(reference='CA')...",0.143378,0.090279,0.022685,0.023844,-0.053098,0.053098,False
24,"C(OriginState_Model, Treatment(reference='CA')...",0.077293,0.028151,0.013028,0.012979,-0.049143,0.049143,False
31,"C(OriginState_Model, Treatment(reference='CA')...",0.013674,0.062067,0.022765,0.022897,0.048393,0.048393,False
45,"C(OriginState_Model, Treatment(reference='CA')...",0.057357,0.010238,0.016578,0.016241,-0.047120,0.047120,False
37,"C(OriginState_Model, Treatment(reference='CA')...",0.150830,0.109743,0.015507,0.015389,-0.041087,0.041087,False
40,"C(OriginState_Model, Treatment(reference='CA')...",0.108701,0.069329,0.030075,0.029983,-0.039372,0.039372,False
46,"C(OriginState_Model, Treatment(reference='CA')...",0.150541,0.112725,0.025559,0.025622,-0.037816,0.037816,False


In [20]:
print(
    f"Sign changes: "
    f"{full_coefficient_stability['Sign_Change'].sum()}"
)

print(
    f"Median absolute difference: "
    f"{full_coefficient_stability['Abs_Estimate_Difference'].median():.6f}"
)

print(
    f"Maximum absolute difference: "
    f"{full_coefficient_stability['Abs_Estimate_Difference'].max():.6f}"
)

Sign changes: 3
Median absolute difference: 0.015720
Maximum absolute difference: 0.066573


### 5.3 Seven-Block Coefficient Stability

In [21]:
reduced_coefficient_stability = pd.DataFrame({
    "Coefficient": reduced_wls_model.params.index,
    "Training_Estimate": reduced_wls_model.params.values,
    "Validation_Estimate": (
        reduced_validation_model.params
        .reindex(reduced_wls_model.params.index)
        .values
    ),
    "Training_SE": reduced_wls_model.bse.values,
    "Validation_SE": (
        reduced_validation_model.bse
        .reindex(reduced_wls_model.params.index)
        .values
    )
})

reduced_coefficient_stability[
    "Estimate_Difference"
] = (
    reduced_coefficient_stability["Validation_Estimate"]
    - reduced_coefficient_stability["Training_Estimate"]
)

reduced_coefficient_stability[
    "Abs_Estimate_Difference"
] = (
    reduced_coefficient_stability[
        "Estimate_Difference"
    ].abs()
)

reduced_coefficient_stability["Sign_Change"] = (
    np.sign(
        reduced_coefficient_stability["Training_Estimate"]
    )
    !=
    np.sign(
        reduced_coefficient_stability["Validation_Estimate"]
    )
)

display(
    reduced_coefficient_stability
    .sort_values(
        "Abs_Estimate_Difference",
        ascending=False
    )
    .head(10)
)

,Coefficient,Training_Estimate,Validation_Estimate,Training_SE,Validation_SE,Estimate_Difference,Abs_Estimate_Difference,Sign_Change
15,"C(RpCarrier_Model, Treatment(reference='AA'))[...",-1.022876,-0.955624,0.040422,0.043193,0.067252,0.067252,False
11,"C(RpCarrier_Model, Treatment(reference='AA'))[...",-0.436566,-0.371274,0.035186,0.037212,0.065293,0.065293,False
17,"C(OriginState_Model, Treatment(reference='CA')...",0.179318,0.117288,0.027400,0.027439,-0.062029,0.062029,False
58,"C(OriginState_Model, Treatment(reference='CA')...",0.148360,0.095200,0.022644,0.023803,-0.053160,0.053160,False
24,"C(OriginState_Model, Treatment(reference='CA')...",0.077112,0.027939,0.013030,0.012981,-0.049173,0.049173,False
31,"C(OriginState_Model, Treatment(reference='CA')...",0.018309,0.066952,0.022733,0.022856,0.048643,0.048643,False
45,"C(OriginState_Model, Treatment(reference='CA')...",0.060888,0.013436,0.016552,0.016217,-0.047453,0.047453,False
37,"C(OriginState_Model, Treatment(reference='CA')...",0.153068,0.112785,0.015493,0.015360,-0.040283,0.040283,False
40,"C(OriginState_Model, Treatment(reference='CA')...",0.115810,0.076207,0.030005,0.029913,-0.039603,0.039603,False
46,"C(OriginState_Model, Treatment(reference='CA')...",0.157591,0.119409,0.025483,0.025552,-0.038182,0.038182,False


In [22]:
print(
    f"Sign changes: "
    f"{reduced_coefficient_stability['Sign_Change'].sum()}"
)

print(
    f"Median absolute difference: "
    f"{reduced_coefficient_stability['Abs_Estimate_Difference'].median():.6f}"
)

print(
    f"Maximum absolute difference: "
    f"{reduced_coefficient_stability['Abs_Estimate_Difference'].max():.6f}"
)

Sign changes: 3
Median absolute difference: 0.015943
Maximum absolute difference: 0.067252


### 5.4 Examine Coefficients with Sign Changes

Sign changes are examined together with coefficient magnitude, standard error, and statistical significance.

A coefficient that is small and statistically insignificant in both samples may change sign simply because its estimated effect is close to zero. Such a change is less concerning than a reversal in a large or statistically meaningful coefficient.

In [23]:
full_sign_change_detail = (
    full_coefficient_stability.loc[
        full_coefficient_stability["Sign_Change"]
    ].copy()
)

full_sign_change_detail["Training_p_value"] = (
    full_sign_change_detail["Coefficient"]
    .map(full_wls_model.pvalues)
)

full_sign_change_detail["Validation_p_value"] = (
    full_sign_change_detail["Coefficient"]
    .map(full_validation_model.pvalues)
)

pd.set_option("display.max_colwidth", None)

display(
    full_sign_change_detail[
        [
            "Coefficient",
            "Training_Estimate",
            "Validation_Estimate",
            "Training_SE",
            "Validation_SE",
            "Training_p_value",
            "Validation_p_value"
        ]
    ]
)

,Coefficient,Training_Estimate,Validation_Estimate,Training_SE,Validation_SE,Training_p_value,Validation_p_value
20,"C(OriginState_Model, Treatment(reference='CA'))[T.AZ]",0.009209,-0.019084,0.013781,0.013823,0.503999,0.167407
41,"C(OriginState_Model, Treatment(reference='CA'))[T.NJ]",0.017306,-0.014013,0.015152,0.015142,0.253403,0.354761
47,"C(OriginState_Model, Treatment(reference='CA'))[T.OR]",0.000493,-0.007368,0.017472,0.017318,0.977487,0.670491


In [24]:
reduced_sign_change_detail = (
    reduced_coefficient_stability.loc[
        reduced_coefficient_stability["Sign_Change"]
    ].copy()
)

reduced_sign_change_detail["Training_p_value"] = (
    reduced_sign_change_detail["Coefficient"]
    .map(reduced_wls_model.pvalues)
)

reduced_sign_change_detail["Validation_p_value"] = (
    reduced_sign_change_detail["Coefficient"]
    .map(reduced_validation_model.pvalues)
)

display(
    reduced_sign_change_detail[
        [
            "Coefficient",
            "Training_Estimate",
            "Validation_Estimate",
            "Training_SE",
            "Validation_SE",
            "Training_p_value",
            "Validation_p_value"
        ]
    ]
)

,Coefficient,Training_Estimate,Validation_Estimate,Training_SE,Validation_SE,Training_p_value,Validation_p_value
20,"C(OriginState_Model, Treatment(reference='CA'))[T.AZ]",0.008951,-0.018950,0.013784,0.013825,0.516105,0.170481
41,"C(OriginState_Model, Treatment(reference='CA'))[T.NJ]",0.016495,-0.014951,0.015155,0.015144,0.276416,0.323545
47,"C(OriginState_Model, Treatment(reference='CA'))[T.OR]",0.002638,-0.005793,0.017468,0.017317,0.879944,0.737990


#### Interpretation

Both candidate specifications show similar coefficient stability across the model-building and validation samples.

Each model contains three sign changes, involving the Arizona, New Jersey, and Oregon origin-state coefficients. These effects are small and statistically insignificant in both samples, indicating that the sign changes reflect estimates fluctuating around zero rather than meaningful reversals in the underlying relationships.

The magnitude of coefficient changes is also very similar across the two candidates. Coefficient stability therefore does not materially distinguish the eight-block and seven-block specifications.

## 6. Training–Validation Comparison

The principal validation results are summarized by comparing the two candidate specifications across the model-building and validation samples.

The table reports coefficient estimates with standard errors together with prediction-error measures and adjusted $R^2$. MSPR is reported for the training-fitted models because it measures their prediction error when applied to the held-out validation observations.

Because both candidates are estimated using WLS, SSE, MSE, and adjusted $R^2$ are calculated from ordinary log-scale prediction errors to provide a common basis for comparison.

### 6.1 Calculate Validation Refit Statistics

In [25]:
full_validation_refit_pred = (
    full_validation_model
    .predict(full_validation_data)
    .to_numpy()
)

reduced_validation_refit_pred = (
    reduced_validation_model
    .predict(reduced_validation_data)
    .to_numpy()
)

full_validation_sse = np.sum(
    (
        validation_actual
        - full_validation_refit_pred
    ) ** 2
)

reduced_validation_sse = np.sum(
    (
        validation_actual
        - reduced_validation_refit_pred
    ) ** 2
)

full_validation_mse = (
    full_validation_sse
    / (len(validation_df) - p_full)
)

reduced_validation_mse = (
    reduced_validation_sse
    / (len(validation_df) - p_reduced)
)

print(
    f"Eight-block validation MSE: "
    f"{full_validation_mse:.6f}"
)

print(
    f"Seven-block validation MSE: "
    f"{reduced_validation_mse:.6f}"
)

Eight-block validation MSE: 0.205866
Seven-block validation MSE: 0.205907


In [26]:
def unweighted_adjusted_r2(
    y,
    y_hat,
    p
):
    n = len(y)

    sse = np.sum(
        (y - y_hat) ** 2
    )

    sst = np.sum(
        (y - np.mean(y)) ** 2
    )

    r_squared = (
        1 - sse / sst
    )

    adjusted_r_squared = (
        1
        - (1 - r_squared)
        * (n - 1)
        / (n - p)
    )

    return adjusted_r_squared

In [27]:
full_train_adj_r2 = unweighted_adjusted_r2(
    train_actual,
    full_train_pred,
    p_full
)

reduced_train_adj_r2 = unweighted_adjusted_r2(
    train_actual,
    reduced_train_pred,
    p_reduced
)

full_validation_adj_r2 = unweighted_adjusted_r2(
    validation_actual,
    full_validation_refit_pred,
    p_full
)

reduced_validation_adj_r2 = unweighted_adjusted_r2(
    validation_actual,
    reduced_validation_refit_pred,
    p_reduced
)

print(
    f"Eight-block adjusted R-squared: "
    f"{full_train_adj_r2:.6f} training, "
    f"{full_validation_adj_r2:.6f} validation"
)

print(
    f"Seven-block adjusted R-squared: "
    f"{reduced_train_adj_r2:.6f} training, "
    f"{reduced_validation_adj_r2:.6f} validation"
)

Eight-block adjusted R-squared: 0.573313 training, 0.568462 validation
Seven-block adjusted R-squared: 0.573235 training, 0.568375 validation


### 6.2 Training–Validation Summary Table

In [28]:
def format_estimate_se(
    estimate,
    standard_error
):
    return (
        f"{estimate:.4f} "
        f"({standard_error:.4f})"
    )

In [29]:
comparison_rows = []

comparison_rows.append({
    "Statistic": "p",
    "Eight-block — Training": str(p_full),
    "Eight-block — Validation": str(p_full),
    "Seven-block — Training": str(p_reduced),
    "Seven-block — Validation": str(p_reduced)
})

for coefficient in full_wls_model.params.index:

    full_train_value = format_estimate_se(
        full_wls_model.params[coefficient],
        full_wls_model.bse[coefficient]
    )

    full_validation_value = format_estimate_se(
        full_validation_model.params[coefficient],
        full_validation_model.bse[coefficient]
    )

    if coefficient in reduced_wls_model.params.index:

        reduced_train_value = format_estimate_se(
            reduced_wls_model.params[coefficient],
            reduced_wls_model.bse[coefficient]
        )

        reduced_validation_value = format_estimate_se(
            reduced_validation_model.params[coefficient],
            reduced_validation_model.bse[coefficient]
        )

    else:
        reduced_train_value = "—"
        reduced_validation_value = "—"

    comparison_rows.append({
        "Statistic": coefficient,
        "Eight-block — Training": full_train_value,
        "Eight-block — Validation": full_validation_value,
        "Seven-block — Training": reduced_train_value,
        "Seven-block — Validation": reduced_validation_value
    })

In [30]:
comparison_rows.extend([
    {
        "Statistic": "SSE",
        "Eight-block — Training":
            f"{full_train_sse:.4f}",
        "Eight-block — Validation":
            f"{full_validation_sse:.4f}",
        "Seven-block — Training":
            f"{reduced_train_sse:.4f}",
        "Seven-block — Validation":
            f"{reduced_validation_sse:.4f}"
    },
    {
        "Statistic": "MSE",
        "Eight-block — Training":
            f"{full_train_mse:.6f}",
        "Eight-block — Validation":
            f"{full_validation_mse:.6f}",
        "Seven-block — Training":
            f"{reduced_train_mse:.6f}",
        "Seven-block — Validation":
            f"{reduced_validation_mse:.6f}"
    },
    {
        "Statistic": "MSPR",
        "Eight-block — Training":
            f"{full_mspr:.6f}",
        "Eight-block — Validation": "—",
        "Seven-block — Training":
            f"{reduced_mspr:.6f}",
        "Seven-block — Validation": "—"
    },
    {
        "Statistic": "Adjusted R-squared",
        "Eight-block — Training":
            f"{full_train_adj_r2:.6f}",
        "Eight-block — Validation":
            f"{full_validation_adj_r2:.6f}",
        "Seven-block — Training":
            f"{reduced_train_adj_r2:.6f}",
        "Seven-block — Validation":
            f"{reduced_validation_adj_r2:.6f}"
    }
])

training_validation_table = pd.DataFrame(
    comparison_rows
)

pd.set_option(
    "display.max_rows",
    None
)

display(training_validation_table)

,Statistic,Eight-block — Training,Eight-block — Validation,Seven-block — Training,Seven-block — Validation
0,p,61,61,60,60
1,Intercept,5.5966 (0.0091),5.6055 (0.0092),5.6072 (0.0087),5.6160 (0.0087)
2,"C(RoundTrip, Treatment(reference=0))[T.1]",0.4354 (0.0056),0.4440 (0.0057),0.4468 (0.0047),0.4552 (0.0048)
3,"C(OnLine, Treatment(reference=1))[T.0]",0.0993 (0.0077),0.0982 (0.0077),0.1129 (0.0067),0.1116 (0.0067)
4,"C(PurchaseWindowGroup, Treatment(reference='21AP'))[T.2290]",-0.3064 (0.0045),-0.3081 (0.0045),-0.3066 (0.0045),-0.3084 (0.0045)
5,"C(PurchaseWindowGroup, Treatment(reference='21AP'))[T.91UP]",-0.3172 (0.0071),-0.3160 (0.0071),-0.3176 (0.0071),-0.3167 (0.0071)
6,"C(RpCarrier_Model, Treatment(reference='AA'))[T.AS]",-0.1087 (0.0102),-0.1338 (0.0105),-0.1113 (0.0102),-0.1359 (0.0105)
7,"C(RpCarrier_Model, Treatment(reference='AA'))[T.B6]",-0.2792 (0.0117),-0.2705 (0.0118),-0.2823 (0.0117),-0.2733 (0.0117)
8,"C(RpCarrier_Model, Treatment(reference='AA'))[T.DL]",0.0625 (0.0070),0.0465 (0.0071),0.0638 (0.0070),0.0478 (0.0071)
9,"C(RpCarrier_Model, Treatment(reference='AA'))[T.F9]",-0.7519 (0.0109),-0.7415 (0.0112),-0.7512 (0.0109),-0.7410 (0.0112)


In [31]:
training_validation_table.to_csv(
    OUTPUT_DIR
    / "07_training_validation_comparison.csv",
    index=False
)

### 6.3 Stability of the Additional Predictor

Because `Coupons` is the only predictor distinguishing the two candidate specifications, its stability across the model-building and validation samples is examined directly.

In [32]:
coupons_validation = pd.DataFrame({
    "Sample": [
        "Model-building",
        "Validation"
    ],
    "Estimate": [
        full_wls_model.params["Coupons"],
        full_validation_model.params["Coupons"]
    ],
    "Standard_Error": [
        full_wls_model.bse["Coupons"],
        full_validation_model.bse["Coupons"]
    ],
    "p_value": [
        full_wls_model.pvalues["Coupons"],
        full_validation_model.pvalues["Coupons"]
    ]
})

display(coupons_validation)

,Sample,Estimate,Standard_Error,p_value
0,Model-building,0.013140,0.003598,0.000260
1,Validation,0.012994,0.003637,0.000354


#### Interpretation

The two candidate specifications perform almost identically across the model-building and validation samples.

The eight-block model has marginally lower training MSE, validation MSE, and MSPR, together with slightly higher adjusted $R^2$ in both samples. However, the numerical differences are very small, indicating that both specifications have similar predictive performance and generalize equally well.

Coefficient stability is also similar across the two candidates. The three sign changes observed in each model involve small and statistically insignificant origin-state effects and therefore do not indicate meaningful instability.

The additional `Coupons` coefficient is highly stable across samples. Its estimate is 0.0131 in the model-building sample and 0.0130 in the validation sample, with similar standard errors of approximately 0.0036. It is statistically significant in both samples, with $p<0.001$.

Thus, although including `Coupons` produces only a marginal improvement in predictive performance, its estimated relationship is stable, statistically supported, and economically interpretable. Since retaining it adds only one parameter, the eight-block specification is preferred despite the slight parsimony advantage of the seven-block alternative.

## 7. Final Model Selection

The full eight-block log-response specification is selected as the final model.

The eight-block and seven-block candidates show almost identical out-of-sample prediction performance and coefficient stability. Parsimony alone would therefore favour the seven-block specification. However, `Coupons`, which captures itinerary complexity through the number of ticketed segments, remains highly stable and statistically significant in both the model-building and validation samples while adding only one parameter.

The eight-block specification also produces marginally lower training MSE, validation MSE, and MSPR, together with slightly higher adjusted $R^2$ in both samples. Although these improvements are small, there is no evidence that retaining `Coupons` harms out-of-sample performance.

Given its stable explanatory contribution, clear interpretation, and minimal additional complexity, `Coupons` is retained and the eight-block specification is preferred.

The final specification includes:

- `TotalDistance`;
- `Coupons`;
- `RoundTrip`;
- `OnLine`;
- `PurchaseWindowGroup`;
- `RpCarrier_Model`;
- `ItinGeoType`;
- `OriginState_Model`.

Refined WLS is retained as the primary estimation approach because it materially reduces the unequal error variance identified during model diagnostics. Because the WLS weights are estimated and some residual heteroskedasticity remains, exact BLUE efficiency cannot be claimed. Nevertheless, the model shows stable coefficient estimates and consistent out-of-sample performance, supporting its use for exploratory explanation and fare benchmarking.

The validated eight-block log-WLS specification is carried forward to Notebook 08 for final estimation using the full analysis sample, economic interpretation, and development of the fare-benchmarking application.